# GO2 本地 MJX 训练

这是 `GO2_train.ipynb` 的本地运行版，用于先复现已经成功的 `mujoco_menagerie/unitree_go2/scene_mjx.xml` 两阶段训练流程。

本 notebook 只做本地化调整：默认使用当前仓库根目录、复用仓库内 `mujoco_menagerie/unitree_go2/`、输出到 `local_training_runs/` 和 `go2_policy_export_local/`。训练 XML、两阶段结构、APG/FoPG 超参数、网络规模、`step_k=13`、`kp=230`、`target_vel=0.75` 都按原 `GO2_train.ipynb` 保持。


## 1. 环境检查

运行这一单元先确认当前 kernel 是 `Python (mjx)`，并确认 MuJoCo、Brax、JAX 与 GO2 MJX XML 可用。


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.8")

PROJECT_DIR = Path.cwd().resolve()
XML_PATH = PROJECT_DIR / "mujoco_menagerie" / "unitree_go2" / "scene_mjx.xml"
POLICY_ROOT = PROJECT_DIR / "go2_policy_export_local"

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import jax
import mujoco
import brax
import mediapy as media
import matplotlib.pyplot as plt

print("project:", PROJECT_DIR)
print("xml_path:", XML_PATH)
print("xml exists:", XML_PATH.exists())
print("python:", sys.version.split()[0])
print("jax:", jax.__version__)
print("mujoco:", mujoco.__version__)
print("brax:", brax.__version__)
print("jax devices:", jax.devices())
print("jax backend:", jax.default_backend())

if not XML_PATH.exists():
    raise FileNotFoundError(XML_PATH)


## 2. 导入本地训练代码

`train_go2_mjx_local.py` 是从原 notebook 拆出的本地脚本入口，保留同一套环境、奖励、两阶段 APG 配置，并补上本地路径、保存、曲线和视频输出。


In [ ]:
from datetime import datetime
from types import SimpleNamespace
from IPython.display import display, Image, Video
import shutil

import importlib
import train_go2_mjx_local as local
local = importlib.reload(local)

# 对齐原 GO2_train.ipynb 的 JAX 设置。
# 注意：debug_nans=True 与 x64=True 会让训练更慢，但这里先按成功 notebook 保持。
args = SimpleNamespace(
    stage="both",
    xml_path=str(XML_PATH),
    run_dir=None,
    policy_root=str(POLICY_ROOT),
    baseline_checkpoint=None,
    seed=0,
    step_k=13,
    servo_kp=230.0,
    target_vel=0.75,
    stage1_updates=499,
    stage2_updates=499,
    stage1_episode_length=240,
    stage2_episode_length=1000,
    horizon_length=32,
    num_envs=64,
    num_eval_envs=64,
    num_evals=10 + 1,
    stage1_lr=1e-4,
    stage2_lr=1.5e-4,
    stage2_schedule_decay=0.995,
    use_float64=True,
    debug_nans=True,
    matmul_precision="high",
    clip_final_action=False,  # 原 notebook 只裁剪 residual，不额外裁剪 baseline+residual。
    render=True,
    render_steps=200,
    render_every=3,
    render_episode_length=1000,
    render_camera=None,
    quick=False,
)

local.configure_jax(args)
local.register_envs()

REUSE_EXISTING_RUN = False
SKIP_STAGE1_IF_CHECKPOINT_EXISTS = False

if REUSE_EXISTING_RUN and "RUN_DIR" in globals():
    RUN_DIR = Path(RUN_DIR)
    print("reuse existing run_dir")
else:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    RUN_DIR = PROJECT_DIR / "local_training_runs" / f"go2_mjx_notebook_{stamp}"
for subdir in ["stage1", "stage2", "checkpoints"]:
    (RUN_DIR / subdir).mkdir(parents=True, exist_ok=True)
args.run_dir = str(RUN_DIR)

print("run_dir:", RUN_DIR)
print("policy_root:", POLICY_ROOT)
print("stage1 hidden:", local.STAGE1_HIDDEN)
print("stage2 hidden:", local.STAGE2_HIDDEN)
print("stage1 updates / episode / horizon / envs / lr:", args.stage1_updates, args.stage1_episode_length, args.horizon_length, args.num_envs, args.stage1_lr)
print("stage2 updates / episode / horizon / envs / lr:", args.stage2_updates, args.stage2_episode_length, args.horizon_length, args.num_envs, args.stage2_lr)


## 3. 轻量验证

这一单元只构建环境并走一步零动作，不会开始训练。确认 observation shape 与 step reward 正常后，再运行训练单元。


In [ ]:
local.validate_setup(args, XML_PATH, baseline_checkpoint=None)


## 4. 第一阶段：trot 模仿学习

对应原 notebook 的第一阶段：`TrotGo2` baseline，APG/FoPG，`hidden_layer_sizes=(256, 128)`，`policy_updates=499`。


In [ ]:
stage1_checkpoint = RUN_DIR / "checkpoints" / "trotting_2hz_policy"
video = RUN_DIR / "stage1" / "stage1_trot_rollout.mp4"
if SKIP_STAGE1_IF_CHECKPOINT_EXISTS and stage1_checkpoint.exists():
    print("found existing stage1 checkpoint, skip retraining:", stage1_checkpoint)
    local.copy_checkpoint(stage1_checkpoint, POLICY_ROOT / "trotting_2hz_policy")
    stage1_make_inference_fn = None
    stage1_params = None
    if args.render and not video.exists():
        print("stage1 video missing; rendering from saved checkpoint")
        stage1_inference_fn, _ = local.make_inference_from_checkpoint(
            checkpoint_path=stage1_checkpoint,
            obs_size=local.BASELINE_OBS_SIZE,
            action_size=local.ACTION_SIZE,
            hidden_layer_sizes=local.STAGE1_HIDDEN,
        )
        stage1_env = local.envs.get_environment(
            "go2_mjx_local_trot",
            xml_path=str(XML_PATH),
            step_k=args.step_k,
            servo_kp=args.servo_kp,
        )
        stage1_demo_env = local.envs.training.EpisodeWrapper(
            stage1_env,
            episode_length=args.render_episode_length,
            action_repeat=1,
        )
        try:
            local.render_rollout_to_video(
                reset_fn=jax.jit(stage1_demo_env.reset),
                step_fn=jax.jit(stage1_demo_env.step),
                inference_fn=stage1_inference_fn,
                env=stage1_demo_env,
                output_path=video,
                n_steps=args.render_steps,
                camera=args.render_camera,
                seed=args.seed + 1,
                render_every=args.render_every,
            )
        except Exception as exc:
            print("stage1 render failed:", exc)
else:
    stage1_checkpoint, stage1_make_inference_fn, stage1_params = local.run_stage1(args, RUN_DIR, XML_PATH)
print("stage1 checkpoint:", stage1_checkpoint)

curve = RUN_DIR / "stage1" / "stage1_training_curve.png"
if curve.exists():
    display(Image(filename=str(curve)))

if video.exists():
    display(Video(str(video), embed=True))


## 5. 第二阶段：前向残差学习

对应原 notebook 的第二阶段：加载第一阶段 baseline，训练 `FwdTrotGo2` residual policy，`hidden_layer_sizes=(128, 64)`，`target_vel=0.75`，`step_k=13`。


In [ ]:
forward_checkpoint = local.run_stage2(args, RUN_DIR, XML_PATH, stage1_checkpoint)
print("forward checkpoint:", forward_checkpoint)

curve = RUN_DIR / "stage2" / "stage2_training_curve.png"
if curve.exists():
    display(Image(filename=str(curve)))

video = RUN_DIR / "stage2" / "stage2_forward_rollout.mp4"
if video.exists():
    display(Video(str(video), embed=True))


## 6. 导出策略

运行完两阶段训练后，策略会同时保存在本次 `RUN_DIR/checkpoints/` 和稳定目录 `go2_policy_export_local/` 下。这个单元再生成一个 zip，便于留档。


In [ ]:
print("run checkpoint dir:", RUN_DIR / "checkpoints")
print("stable policy root:", POLICY_ROOT)

zip_base = PROJECT_DIR / "go2_policy_export_local"
zip_path = shutil.make_archive(str(zip_base), "zip", POLICY_ROOT)
print("zip:", zip_path)


## 7. 加载前向策略做一次推理和渲染

用于确认保存后的第二阶段策略可以重新加载，并能输出 12 维动作。


In [ ]:
loaded_inference_fn, loaded_params = local.make_inference_from_checkpoint(
    checkpoint_path=POLICY_ROOT / "forward_locomotion_policy",
    obs_size=local.FORWARD_OBS_SIZE,
    action_size=local.ACTION_SIZE,
    hidden_layer_sizes=local.STAGE2_HIDDEN,
)

baseline_inference_fn, _ = local.make_inference_from_checkpoint(
    checkpoint_path=POLICY_ROOT / "trotting_2hz_policy",
    obs_size=local.BASELINE_OBS_SIZE,
    action_size=local.ACTION_SIZE,
    hidden_layer_sizes=local.STAGE1_HIDDEN,
)

deploy_env = local.envs.get_environment(
    "go2_mjx_local_forward",
    baseline_inference_fn=baseline_inference_fn,
    xml_path=str(XML_PATH),
    target_vel=args.target_vel,
    step_k=args.step_k,
    servo_kp=args.servo_kp,
    clip_final_action=args.clip_final_action,
)
deploy_demo_env = local.envs.training.EpisodeWrapper(deploy_env, episode_length=400, action_repeat=1)
state = deploy_demo_env.reset(jax.random.PRNGKey(0))
action, _ = loaded_inference_fn(state.obs, jax.random.PRNGKey(1))
print("action shape:", action.shape)
print("first 4 action dims:", action[:4])

reload_video = RUN_DIR / "stage2" / "stage2_reload_rollout.mp4"
local.render_rollout_to_video(
    reset_fn=jax.jit(deploy_demo_env.reset),
    step_fn=jax.jit(deploy_demo_env.step),
    inference_fn=loaded_inference_fn,
    env=deploy_demo_env,
    output_path=reload_video,
    n_steps=200,
    camera="track",
    seed=3,
    render_every=args.render_every,
)
display(Video(str(reload_video), embed=True))


## 命令行备用

如果后面需要脱离 notebook 批量跑同一套配置，可以在终端执行：

```bash
conda run -n mjx python train_go2_mjx_local.py --stage both --render
```

当前 CLI 默认已经按 notebook 行为设置：`jax_debug_nans=True`、`use_float64=True`、第二阶段不额外裁剪 `baseline+residual`。
